In [1]:
import os
import findspark

os.environ["SPARK_HOME"] = "/workspaces/Stream_Analytics-Group_7/spark-4.0.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

findspark.init()

import pyspark
print("PySpark version:", pyspark.__version__)

PySpark version: 4.0.1


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("group7-analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 14:59:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/19 14:59:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.0.1


In [6]:
from pathlib import Path

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

bronze_orders_path = repo_root / "data" / "bronze" / "orders"
bronze_delivery_path = repo_root / "data" / "bronze" / "delivery"

print("Repo root:", repo_root)
print("Orders bronze path:", bronze_orders_path)
print("Delivery bronze path:", bronze_delivery_path)
print("Orders exists:", bronze_orders_path.exists())
print("Delivery exists:", bronze_delivery_path.exists())

Repo root: /workspaces/Stream_Analytics-Group_7
Orders bronze path: /workspaces/Stream_Analytics-Group_7/data/bronze/orders
Delivery bronze path: /workspaces/Stream_Analytics-Group_7/data/bronze/delivery
Orders exists: True
Delivery exists: True


In [7]:
orders = spark.read.parquet(str(bronze_orders_path))
delivery = spark.read.parquet(str(bronze_delivery_path))

print("Orders:", orders.count())
print("Delivery:", delivery.count())

orders.show(5, truncate=False)
delivery.show(5, truncate=False)

Orders: 94
Delivery: 738


+------------------------------------+------------------------------------+----------------+-----------------------+-----------------------+-------------+-------+----------+-----------+--------+--------------------------+-------+-----------------------+-----------------------+
|event_id                            |order_id                            |event_type      |event_ts               |ingest_ts              |restaurant_id|zone_id|courier_id|order_value|is_promo|estimated_delivery_minutes|version|event_time             |ingest_time            |
+------------------------------------+------------------------------------+----------------+-----------------------+-----------------------+-------------+-------+----------+-----------+--------+--------------------------+-------+-----------------------+-----------------------+
|64dce1a6-2081-43d3-a08e-37e1a9b9aa4d|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|prep_started    |2026-04-16 20:19:24.093|2026-04-16 20:36:46.102|rest_20      |east   |NULL

In [8]:
from pyspark.sql.functions import (
    col,
    avg,
    when,
    count,
    unix_timestamp,
    expr
)

In [9]:
orders_created = (
    orders
    .filter(col("event_type") == "created")
    .select(
        col("order_id"),
        col("zone_id"),
        col("event_time").alias("created_time")
    )
)

orders_delivered = (
    orders
    .filter(col("event_type") == "delivered")
    .select(
        col("order_id"),
        col("event_time").alias("delivered_time")
    )
)

orders_created.show(5, truncate=False)
orders_delivered.show(5, truncate=False)

+------------------------------------+-------+-----------------------+
|order_id                            |zone_id|created_time           |
+------------------------------------+-------+-----------------------+
|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|east   |2026-04-16 20:17:24.093|
|fa98b2a0-18a7-49e8-ac64-84b64af63887|center |2026-04-16 20:17:45.093|
|91dccec2-f176-4852-ad1a-184101a10be8|west   |2026-04-16 20:17:23.093|
|b1e3d104-3943-4672-81bd-a719b5cfef2f|east   |2026-04-16 20:17:07.093|
|180f31da-cea8-4d27-9685-71d6184cc540|west   |2026-04-16 20:17:02.093|
+------------------------------------+-------+-----------------------+
only showing top 5 rows
+------------------------------------+-----------------------+
|order_id                            |delivered_time         |
+------------------------------------+-----------------------+
|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|2026-04-16 20:54:24.093|
|b1e3d104-3943-4672-81bd-a719b5cfef2f|2026-04-16 20:50:07.093|
|180f31da-cea8-4d27-96

In [10]:
delivery_time_per_order = (
    orders_created
    .join(orders_delivered, on="order_id", how="inner")
)

delivery_time_per_order.show(5, truncate=False)

+------------------------------------+-------+-----------------------+-----------------------+
|order_id                            |zone_id|created_time           |delivered_time         |
+------------------------------------+-------+-----------------------+-----------------------+
|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|east   |2026-04-16 20:17:24.093|2026-04-16 20:54:24.093|
|b1e3d104-3943-4672-81bd-a719b5cfef2f|east   |2026-04-16 20:17:07.093|2026-04-16 20:50:07.093|
|180f31da-cea8-4d27-9685-71d6184cc540|west   |2026-04-16 20:17:02.093|2026-04-16 20:48:02.093|
|53892555-8e52-43a3-acd6-1ace65a62c66|south  |2026-04-16 20:17:12.093|2026-04-16 20:48:12.093|
|2f433945-b53d-401b-b759-d0be390b09e4|west   |2026-04-16 20:18:15.093|2026-04-16 21:04:15.093|
+------------------------------------+-------+-----------------------+-----------------------+
only showing top 5 rows


In [11]:
delivery_time_per_order = delivery_time_per_order.withColumn(
    "delivery_minutes",
    (unix_timestamp("delivered_time") - unix_timestamp("created_time")) / 60.0
)

delivery_time_per_order.select(
    "order_id", "zone_id", "created_time", "delivered_time", "delivery_minutes"
).show(10, truncate=False)

+------------------------------------+-------+-----------------------+-----------------------+----------------+
|order_id                            |zone_id|created_time           |delivered_time         |delivery_minutes|
+------------------------------------+-------+-----------------------+-----------------------+----------------+
|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|east   |2026-04-16 20:17:24.093|2026-04-16 20:54:24.093|37.0            |
|b1e3d104-3943-4672-81bd-a719b5cfef2f|east   |2026-04-16 20:17:07.093|2026-04-16 20:50:07.093|33.0            |
|180f31da-cea8-4d27-9685-71d6184cc540|west   |2026-04-16 20:17:02.093|2026-04-16 20:48:02.093|31.0            |
|53892555-8e52-43a3-acd6-1ace65a62c66|south  |2026-04-16 20:17:12.093|2026-04-16 20:48:12.093|31.0            |
|2f433945-b53d-401b-b759-d0be390b09e4|west   |2026-04-16 20:18:15.093|2026-04-16 21:04:15.093|46.0            |
|4c25e5b5-5664-43fb-9a0b-d506e5c56025|center |2026-04-16 20:18:00.093|2026-04-16 20:57:00.093|39.0      

In [12]:
delivery_time_clean = (
    delivery_time_per_order
    .filter(col("delivery_minutes") >= 0)
    .filter(col("delivery_minutes") <= 180)
)

delivery_time_clean.show(10, truncate=False)

+------------------------------------+-------+-----------------------+-----------------------+----------------+
|order_id                            |zone_id|created_time           |delivered_time         |delivery_minutes|
+------------------------------------+-------+-----------------------+-----------------------+----------------+
|f750a99f-fa2c-45e0-b0c3-1da8d120fb87|east   |2026-04-16 20:17:24.093|2026-04-16 20:54:24.093|37.0            |
|b1e3d104-3943-4672-81bd-a719b5cfef2f|east   |2026-04-16 20:17:07.093|2026-04-16 20:50:07.093|33.0            |
|180f31da-cea8-4d27-9685-71d6184cc540|west   |2026-04-16 20:17:02.093|2026-04-16 20:48:02.093|31.0            |
|53892555-8e52-43a3-acd6-1ace65a62c66|south  |2026-04-16 20:17:12.093|2026-04-16 20:48:12.093|31.0            |
|2f433945-b53d-401b-b759-d0be390b09e4|west   |2026-04-16 20:18:15.093|2026-04-16 21:04:15.093|46.0            |
|4c25e5b5-5664-43fb-9a0b-d506e5c56025|center |2026-04-16 20:18:00.093|2026-04-16 20:57:00.093|39.0      

In [13]:
zone_delivery_stats = (
    delivery_time_clean
    .groupBy("zone_id")
    .agg(
        avg("delivery_minutes").alias("avg_delivery_minutes"),
        count("*").alias("delivered_orders")
    )
)

zone_delivery_stats.show(truncate=False)

+-------+--------------------+----------------+
|zone_id|avg_delivery_minutes|delivered_orders|
+-------+--------------------+----------------+
|east   |31.666666666666668  |3               |
|west   |36.666666666666664  |3               |
|center |39.57142857142857   |7               |
|south  |35.5                |2               |
+-------+--------------------+----------------+



In [14]:
zone_anomalies = (
    zone_delivery_stats
    .withColumn(
        "anomaly_flag",
        when(col("avg_delivery_minutes") > 45, 1).otherwise(0)
    )
)

zone_anomalies.show(truncate=False)

+-------+--------------------+----------------+------------+
|zone_id|avg_delivery_minutes|delivered_orders|anomaly_flag|
+-------+--------------------+----------------+------------+
|east   |31.666666666666668  |3               |0           |
|west   |36.666666666666664  |3               |0           |
|center |39.57142857142857   |7               |0           |
|south  |35.5                |2               |0           |
+-------+--------------------+----------------+------------+



In [15]:
zone_anomalies.filter(col("anomaly_flag") == 1).show(truncate=False)

+-------+--------------------+----------------+------------+
|zone_id|avg_delivery_minutes|delivered_orders|anomaly_flag|
+-------+--------------------+----------------+------------+
+-------+--------------------+----------------+------------+



In [16]:
zone_anomalies.orderBy(col("avg_delivery_minutes").desc()).show(truncate=False)

+-------+--------------------+----------------+------------+
|zone_id|avg_delivery_minutes|delivered_orders|anomaly_flag|
+-------+--------------------+----------------+------------+
|center |39.57142857142857   |7               |0           |
|west   |36.666666666666664  |3               |0           |
|south  |35.5                |2               |0           |
|east   |31.666666666666668  |3               |0           |
+-------+--------------------+----------------+------------+



In [17]:
extreme_orders = (
    delivery_time_clean
    .withColumn(
        "extreme_flag",
        when(col("delivery_minutes") > 60, 1).otherwise(0)
    )
    .filter(col("extreme_flag") == 1)
)

extreme_orders.select("order_id", "zone_id", "delivery_minutes").show(truncate=False)

+--------+-------+----------------+
|order_id|zone_id|delivery_minutes|
+--------+-------+----------------+
+--------+-------+----------------+



In [19]:
zone_p95 = (
    delivery_time_clean
    .groupBy("zone_id")
    .agg(
        expr("percentile_approx(delivery_minutes, 0.95)").alias("p95_delivery_minutes")
    )
)

zone_p95.show(truncate=False)

+-------+--------------------+
|zone_id|p95_delivery_minutes|
+-------+--------------------+
|east   |37.0                |
|west   |46.0                |
|center |45.0                |
|south  |40.0                |
+-------+--------------------+



In [20]:
zone_anomalies_enriched = (
    zone_anomalies
    .join(zone_p95, on="zone_id", how="left")
    .orderBy(col("avg_delivery_minutes").desc())
)

zone_anomalies_enriched.show(truncate=False)

+-------+--------------------+----------------+------------+--------------------+
|zone_id|avg_delivery_minutes|delivered_orders|anomaly_flag|p95_delivery_minutes|
+-------+--------------------+----------------+------------+--------------------+
|center |39.57142857142857   |7               |0           |45.0                |
|west   |36.666666666666664  |3               |0           |46.0                |
|south  |35.5                |2               |0           |40.0                |
|east   |31.666666666666668  |3               |0           |37.0                |
+-------+--------------------+----------------+------------+--------------------+

